In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# On Kaggle, input data lives somewhere under /kaggle/input (read-only), nested
# under a competition-specific subfolder. Locally (e.g. VS Code), it's flat in ./data,
# populated by `kaggle competitions download -c titanic`. Either way, find the folder
# that actually contains train.csv rather than hardcoding the path.

import os

SEARCH_ROOT = '/kaggle/input' if os.path.isdir('/kaggle/input') else 'data'
WORKING_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else 'output'

INPUT_DIR = None
for dirname, _, filenames in os.walk(SEARCH_ROOT):
    for filename in filenames:
        print(os.path.join(dirname, filename))
    if 'train.csv' in filenames:
        INPUT_DIR = dirname

if INPUT_DIR is None:
    raise FileNotFoundError(f'Could not find train.csv under {SEARCH_ROOT}')

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

data/test.csv
data/train.csv
data/gender_submission.csv


## Load data

In [2]:
train = pd.read_csv(os.path.join(INPUT_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(INPUT_DIR, 'test.csv'))

print(train.shape, test.shape)
train.head()

(891, 12) (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Feature engineering

- `Title` extracted from `Name` (Mr/Mrs/Miss/Master/Rare) tends to capture age/sex/social status signal better than `Name` itself.
- `FamilySize` = `SibSp` + `Parch` + 1, and `IsAlone` as a flag, since family size affected survival odds.
- Missing `Age`/`Fare`/`Embarked` are imputed inside the model pipeline so train/test are handled identically.

In [3]:
def engineer_features(df):
    df = df.copy()

    df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.')
    title_map = {
        'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
        'Lady': 'Rare', 'Countess': 'Rare', 'Capt': 'Rare', 'Col': 'Rare',
        'Don': 'Rare', 'Dr': 'Rare', 'Major': 'Rare', 'Rev': 'Rare',
        'Sir': 'Rare', 'Jonkheer': 'Rare', 'Dona': 'Rare',
    }
    df['Title'] = df['Title'].replace(title_map)
    df.loc[~df['Title'].isin(['Mr', 'Mrs', 'Miss', 'Master', 'Rare']), 'Title'] = 'Rare'

    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

    return df

train_fe = engineer_features(train)
test_fe = engineer_features(test)

train_fe[['Name', 'Title', 'FamilySize', 'IsAlone']].head()

,Name,Title,FamilySize,IsAlone
0,"Braund, Mr. Owen Harris",Mr,2,0
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs,2,0
2,"Heikkinen, Miss. Laina",Miss,1,1
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs,2,0
4,"Allen, Mr. William Henry",Mr,1,1


## Build preprocessing + model pipeline

Numeric features are median-imputed and scaled; categorical features are mode-imputed and one-hot encoded. Wrapping this in a `Pipeline` means the same transforms are learned on train and applied to test without leakage.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

numeric_features = ['Age', 'Fare', 'FamilySize', 'IsAlone']
categorical_features = ['Pclass', 'Sex', 'Embarked', 'Title']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

X = train_fe[numeric_features + categorical_features]
y = train_fe['Survived']
X_test = test_fe[numeric_features + categorical_features]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Compare a few models with cross-validation

5-fold stratified cross-validation accuracy gives a more honest estimate than a single train/test split, since the dataset is small (~890 rows).

In [5]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'RandomForest': RandomForestClassifier(n_estimators=300, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(random_state=42),
}

results = {}
for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    scores = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy')
    results[name] = scores
    print(f'{name:18s} mean={scores.mean():.4f}  std={scores.std():.4f}')

LogisticRegression mean=0.8272  std=0.0065


RandomForest       mean=0.8103  std=0.0161


GradientBoosting   mean=0.8395  std=0.0068


## Train final model and write submission

In [6]:
best_name = max(results, key=lambda name: results[name].mean())
print(f'Best model: {best_name}')

final_pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', models[best_name])])
final_pipe.fit(X, y)

predictions = final_pipe.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test_fe['PassengerId'],
    'Survived': predictions,
})
os.makedirs(WORKING_DIR, exist_ok=True)
submission.to_csv(os.path.join(WORKING_DIR, 'submission.csv'), index=False)
submission.head()

Best model: GradientBoosting


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
